# Rebuild the Best Readmission Models

This notebook starts from the cleaned dataset and rebuilds the three final models used in the project:
Logistic Regression, XGBoost, and Random Forest.

To keep the workflow reviewer-friendly, the notebook does **not** run hyperparameter tuning again. Instead, it uses the previously selected best hyperparameters and previously saved feature subsets directly inside the notebook.


## Notebook Roadmap

The workflow in this notebook is intentionally simple:

1. Upload and load the cleaned dataset.
2. Reuse the saved best hyperparameters.
3. Reuse the saved feature lists for each model.
4. Rebuild and fit the three final models.
5. Compare their test-set performance in one summary table.


## 1. Upload the Cleaned Dataset in Colab


In [ ]:
from pathlib import Path

# In Google Colab, upload only the cleaned dataset file:
# diabetes_final_ml_dataset_encoded.csv
try:
    from google.colab import files
    uploaded = files.upload()
    print('Uploaded files:', list(uploaded.keys()))
except ImportError:
    print('If you are not using Colab, place diabetes_final_ml_dataset_encoded.csv in the current working directory.')

DATA_PATH = Path('diabetes_final_ml_dataset_encoded.csv')
OUTPUT_DIR = Path('model_rebuild_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Dataset available:', DATA_PATH.exists())


## 2. Import Libraries


In [ ]:
import ast
import re
import warnings

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')


## 3. Reuse the Saved Hyperparameters and Feature Lists


In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET_COL = 'readmitted_30d'

# These are the final best hyperparameters selected in the earlier CPU training pipeline.
best_params_map = {
    'Logistic Regression + RFE': {'penalty': 'l2', 'C': 0.001},
    'XGBoost CPU + RFE': {
        'subsample': 0.8,
        'reg_lambda': 1,
        'reg_alpha': 0,
        'n_estimators': 200,
        'min_child_weight': 3,
        'max_depth': 4,
        'learning_rate': 0.05,
        'gamma': 0.1,
        'colsample_bytree': 0.7,
    },
    'Random Forest + RFE': {
        'n_estimators': 150,
        'min_samples_split': 20,
        'min_samples_leaf': 8,
        'max_features': 'sqrt',
        'max_depth': 12,
        'bootstrap': False,
    },
}

# These are the final RFE-selected feature subsets saved from the earlier training workflow.
selected_features_map = {
    'Logistic Regression + RFE': [
        'time_in_hospital', 'num_procedures', 'num_medications', 'number_outpatient',
        'number_inpatient', 'number_diagnoses', 'diabetes_medication_used', 'age_numeric',
        'hba1c_tested', 'prior_utilization_total', 'metformin_No', 'metformin_Steady',
        'metformin_Up', 'nateglinide_Up', 'glipizide_No', 'glipizide_Steady',
        'rosiglitazone_No', 'rosiglitazone_Steady', 'insulin_No', 'insulin_Steady',
        'glyburide-metformin_Up', 'glipizide-metformin_Steady',
        'discharge_group_Nursing_Facility_or_Transfer', 'discharge_group_Other',
        'diag_1_group_Digestive', 'diag_1_group_Genitourinary', 'diag_1_group_Musculoskeletal',
        'diag_1_group_Other', 'diag_1_group_Respiratory', 'diag_2_group_Diabetes',
        'diag_2_group_Neoplasms', 'diag_3_group_Diabetes', 'diag_3_group_Digestive',
        'diag_3_group_Genitourinary', 'medical_specialty_grouped_Family/GeneralPractice',
        'medical_specialty_grouped_InternalMedicine', 'medical_specialty_grouped_Missing',
        'medical_specialty_grouped_Nephrology', 'medical_specialty_grouped_ObstetricsandGynecology',
        'medical_specialty_grouped_Surgery-Cardiovascular/Thoracic'
    ],
    'XGBoost CPU + RFE': [
        'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications',
        'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses',
        'diabetes_medication_used', 'age_numeric', 'hba1c_tested', 'prior_utilization_total',
        'hba1c_high', 'metformin_No', 'metformin_Steady', 'glipizide_No', 'insulin_No',
        'insulin_Steady', 'discharge_group_Nursing_Facility_or_Transfer', 'discharge_group_Other',
        'admission_type_group_Emergency_Urgent', 'admission_type_group_Other_Unknown',
        'admission_source_group_Referral', 'admission_source_group_Transfer_Other',
        'diag_1_group_Diabetes', 'diag_1_group_Digestive', 'diag_1_group_Genitourinary',
        'diag_1_group_Musculoskeletal', 'diag_1_group_Other', 'diag_1_group_Respiratory',
        'diag_2_group_Diabetes', 'diag_2_group_Injury', 'diag_2_group_Neoplasms',
        'diag_2_group_Respiratory', 'diag_3_group_Genitourinary', 'diag_3_group_Missing',
        'diag_3_group_Neoplasms', 'medical_specialty_grouped_ObstetricsandGynecology',
        'medical_specialty_grouped_Orthopedics-Reconstructive',
        'medical_specialty_grouped_Surgery-Cardiovascular/Thoracic'
    ],
    'Random Forest + RFE': [
        'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications',
        'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses',
        'is_male', 'medication_changed', 'diabetes_medication_used', 'age_numeric',
        'hba1c_tested', 'prior_utilization_total', 'race_Caucasian', 'A1Cresult_Not_Tested',
        'metformin_No', 'metformin_Steady', 'glipizide_No', 'glipizide_Steady',
        'glyburide_No', 'insulin_No', 'insulin_Steady',
        'discharge_group_Nursing_Facility_or_Transfer', 'discharge_group_Other',
        'admission_type_group_Emergency_Urgent', 'admission_source_group_Referral',
        'admission_source_group_Transfer_Other', 'diag_1_group_Diabetes', 'diag_1_group_Other',
        'diag_1_group_Respiratory', 'diag_2_group_Diabetes', 'diag_2_group_Genitourinary',
        'diag_2_group_Other', 'diag_2_group_Respiratory', 'diag_3_group_Diabetes',
        'diag_3_group_Other', 'medical_specialty_grouped_Family/GeneralPractice',
        'medical_specialty_grouped_InternalMedicine', 'medical_specialty_grouped_Missing'
    ],
}

best_params_df = pd.DataFrame(
    [
        {'Model': model_name, 'Best Params': str(params)}
        for model_name, params in best_params_map.items()
    ]
)
display(best_params_df)

for model_name, features in selected_features_map.items():
    print(model_name, '->', len(features), 'features')


## 4. Load the Cleaned Dataset and Create the Train/Test Split


In [ ]:
# Load the final cleaned and encoded dataset produced in the preprocessing notebook.
df = pd.read_csv(DATA_PATH)

print('Dataset shape:', df.shape)
print('Missing values:', df.isna().sum().sum())
print('Duplicate columns:', df.columns[df.columns.duplicated()].tolist())

assert TARGET_COL in df.columns, 'Target column not found in the cleaned dataset.'
assert df.isna().sum().sum() == 0, 'The cleaned dataset still contains missing values.'

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('scale_pos_weight:', round(scale_pos_weight, 4))
print('\nTarget distribution:')
display(y.value_counts().to_frame(name='count').assign(rate=y.value_counts(normalize=True).round(4)))


## 5. Define a Shared Evaluation Function


In [ ]:
def evaluate_model(model_name, model, X_test_eval, y_test_eval, selected_features, threshold=0.5):
    """Evaluate a fitted binary classifier on the held-out test set."""
    y_proba = model.predict_proba(X_test_eval)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test_eval, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    flagged = tp + fp

    result = {
        'Model': model_name,
        'Selected Features': len(selected_features),
        'ROC-AUC': roc_auc_score(y_test_eval, y_proba),
        'PR-AUC': average_precision_score(y_test_eval, y_proba),
        'Accuracy': accuracy_score(y_test_eval, y_pred),
        'Balanced Accuracy': balanced_accuracy_score(y_test_eval, y_pred),
        'Precision': precision_score(y_test_eval, y_pred, zero_division=0),
        'Recall': recall_score(y_test_eval, y_pred, zero_division=0),
        'Specificity': specificity,
        'F1-score': f1_score(y_test_eval, y_pred, zero_division=0),
        'TP': tp,
        'FP': fp,
        'FN': fn,
        'TN': tn,
        'Patients Flagged': flagged,
        'Flagged Rate %': flagged / len(y_test_eval) * 100,
    }

    print(f'\n{model_name}')
    print(classification_report(y_test_eval, y_pred))
    print('Confusion Matrix:')
    print(confusion_matrix(y_test_eval, y_pred))

    return result, y_proba, y_pred

all_results = []
prediction_store = {'actual': y_test.values}


## 6. Rebuild Logistic Regression with the Saved Best Settings


In [ ]:
# Standardize the full feature matrix because Logistic Regression is scale-sensitive.
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index,
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index,
)

# Keep only the previously selected RFE features.
lr_features = selected_features_map['Logistic Regression + RFE']
X_train_lr = X_train_scaled[lr_features]
X_test_lr = X_test_scaled[lr_features]

# Rebuild the final Logistic Regression model using the stored best hyperparameters.
lr_params = best_params_map['Logistic Regression + RFE'].copy()
best_lr_model = LogisticRegression(
    class_weight='balanced',
    solver='liblinear',
    max_iter=5000,
    random_state=RANDOM_STATE,
    **lr_params,
)
best_lr_model.fit(X_train_lr, y_train)

lr_result, lr_proba, lr_pred = evaluate_model(
    'Logistic Regression + RFE',
    best_lr_model,
    X_test_lr,
    y_test,
    lr_features,
)

all_results.append(lr_result)
prediction_store['lr_probability'] = lr_proba
prediction_store['lr_prediction'] = lr_pred


## 7. Rebuild XGBoost with the Saved Best Settings


In [ ]:
# Keep only the previously selected XGBoost feature subset.
xgb_features = selected_features_map['XGBoost CPU + RFE']
X_train_xgb = X_train[xgb_features]
X_test_xgb = X_test[xgb_features]

# Rebuild the final XGBoost model using the stored best hyperparameters.
xgb_params = best_params_map['XGBoost CPU + RFE'].copy()
best_xgb_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    scale_pos_weight=scale_pos_weight,
    tree_method='hist',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    **xgb_params,
)
best_xgb_model.fit(X_train_xgb, y_train)

xgb_result, xgb_proba, xgb_pred = evaluate_model(
    'XGBoost CPU + RFE',
    best_xgb_model,
    X_test_xgb,
    y_test,
    xgb_features,
)

all_results.append(xgb_result)
prediction_store['xgb_probability'] = xgb_proba
prediction_store['xgb_prediction'] = xgb_pred


## 8. Rebuild Random Forest with the Saved Best Settings


In [ ]:
# Keep only the previously selected Random Forest feature subset.
rf_features = selected_features_map['Random Forest + RFE']
X_train_rf = X_train[rf_features]
X_test_rf = X_test[rf_features]

# Rebuild the final Random Forest model using the stored best hyperparameters.
rf_params = best_params_map['Random Forest + RFE'].copy()
best_rf_model = RandomForestClassifier(
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    **rf_params,
)
best_rf_model.fit(X_train_rf, y_train)

rf_result, rf_proba, rf_pred = evaluate_model(
    'Random Forest + RFE',
    best_rf_model,
    X_test_rf,
    y_test,
    rf_features,
)

all_results.append(rf_result)
prediction_store['rf_probability'] = rf_proba
prediction_store['rf_prediction'] = rf_pred


## 9. Build the Final Comparison Table


In [ ]:
# Collect the main performance metrics in one comparison table.
model_results = pd.DataFrame(all_results)

metric_cols = [
    'Model',
    'Selected Features',
    'ROC-AUC',
    'PR-AUC',
    'Accuracy',
    'Balanced Accuracy',
    'Precision',
    'Recall',
    'Specificity',
    'F1-score',
    'TP',
    'FP',
    'FN',
    'TN',
    'Patients Flagged',
    'Flagged Rate %',
]

model_results = model_results[metric_cols].round(4)
display(model_results)


## 10. Save the Notebook Outputs


In [ ]:
# Save the comparison table and test-set predictions generated in this notebook.
model_results.to_csv(
    OUTPUT_DIR / 'rebuild_model_comparison.csv',
    index=False,
)

predictions_df = pd.DataFrame(prediction_store)
predictions_df.to_csv(
    OUTPUT_DIR / 'rebuild_test_set_predictions.csv',
    index=False,
)

print('Saved notebook outputs to:', OUTPUT_DIR)
print('- rebuild_model_comparison.csv')
print('- rebuild_test_set_predictions.csv')


1. Evaluation plots for selected XGBoost model

In [ ]:
# ============================================================
# XGBoost Evaluation Diagnostic Plots
# ROC Curve, Precision-Recall Curve, Calibration Curve, Confusion Matrix
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    brier_score_loss
)

from sklearn.calibration import calibration_curve

# ------------------------------------------------------------
# Make sure probability predictions exist
# ------------------------------------------------------------

xgb_proba = best_xgb_model.predict_proba(X_test_xgb)[:, 1]

# Default threshold
threshold = 0.5
xgb_pred = (xgb_proba >= threshold).astype(int)

2. ROC Curve

In [ ]:
# ============================================================
# ROC Curve
# ============================================================

fpr, tpr, roc_thresholds = roc_curve(y_test, xgb_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")

plt.title("ROC Curve - XGBoost Tuned")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(OUTPUT_DIR / "xgboost_roc_curve.png", dpi=300, bbox_inches="tight")
plt.show()

3. Precision-Recall Curve

In [ ]:
# ============================================================
# Precision-Recall Curve
# ============================================================

precision, recall, pr_thresholds = precision_recall_curve(y_test, xgb_proba)
pr_auc = average_precision_score(y_test, xgb_proba)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, label=f"PR-AUC = {pr_auc:.3f}")

plt.title("Precision-Recall Curve - XGBoost Tuned")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend(loc="upper right")
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(OUTPUT_DIR / "xgboost_precision_recall_curve.png", dpi=300, bbox_inches="tight")
plt.show()

4. Calibration Curve

In [ ]:
# ============================================================
# Calibration Curve
# ============================================================

brier = brier_score_loss(y_test, xgb_proba)

prob_true, prob_pred = calibration_curve(
    y_test,
    xgb_proba,
    n_bins=10,
    strategy="quantile"
)

plt.figure(figsize=(8, 6))
plt.plot(prob_pred, prob_true, marker="o", label="Model")
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")

plt.title(f"Calibration Curve - Brier {brier:.3f}")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed readmission rate")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(OUTPUT_DIR / "xgboost_calibration_curve.png", dpi=300, bbox_inches="tight")
plt.show()

5. Confusion Matrix at threshold 0.5

In [ ]:
# ============================================================
# Confusion Matrix
# ============================================================

cm = confusion_matrix(y_test, xgb_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[0, 1],
    yticklabels=[0, 1]
)

plt.title(f"Confusion Matrix - Threshold {threshold}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()

plt.savefig(OUTPUT_DIR / "xgboost_confusion_matrix_threshold_05.png", dpi=300, bbox_inches="tight")
plt.show()

tn, fp, fn, tp = cm.ravel()

print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

6. Threshold comparison table

In [ ]:
# ============================================================
# Threshold Comparison Table
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)

thresholds = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]

threshold_rows = []

for t in thresholds:
    y_pred_t = (xgb_proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()

    threshold_rows.append({
        "Threshold": t,
        "Accuracy": accuracy_score(y_test, y_pred_t),
        "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred_t),
        "Precision": precision_score(y_test, y_pred_t, zero_division=0),
        "Recall": recall_score(y_test, y_pred_t, zero_division=0),
        "F1-score": f1_score(y_test, y_pred_t, zero_division=0),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "Patients Flagged": tp + fp,
        "Flagged Rate %": (tp + fp) / len(y_test) * 100
    })

threshold_df = pd.DataFrame(threshold_rows).round(4)

display(threshold_df)

threshold_df.to_csv(
    OUTPUT_DIR / "xgboost_threshold_comparison.csv",
    index=False
)

In [ ]:
# ============================================================
# Precision / Recall / F1 by Threshold
# ============================================================

plt.figure(figsize=(8, 6))

plt.plot(threshold_df["Threshold"], threshold_df["Precision"], marker="o", label="Precision")
plt.plot(threshold_df["Threshold"], threshold_df["Recall"], marker="o", label="Recall")
plt.plot(threshold_df["Threshold"], threshold_df["F1-score"], marker="o", label="F1-score")

plt.title("Precision, Recall, and F1 by Threshold - XGBoost")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(OUTPUT_DIR / "xgboost_threshold_metric_tradeoff.png", dpi=300, bbox_inches="tight")
plt.show()

### Model Evaluation Interpretation

The XGBoost model achieved moderate discriminative performance, with ROC-AUC around 0.677 and PR-AUC around 0.228. Because the positive readmission class is highly imbalanced, PR-AUC, recall, and the confusion matrix are more informative than accuracy alone.

At the default threshold of 0.5, the model identified 1,283 true 30-day readmissions but missed 980 readmitted patients. This shows that the model is useful for patient triage, but the classification threshold should be treated as a business and clinical decision rather than a fixed technical value.

The calibration curve suggests that the model is not perfectly calibrated and tends to overestimate absolute readmission probabilities. Therefore, the model should currently be interpreted primarily as a risk-ranking tool rather than a precise probability forecasting tool.

Before deployment, different thresholds should be compared to balance recall, false positives, and discharge planner workload.

## 11. Probability Calibration and Final Operating Threshold

The previous evaluation showed that the XGBoost model has useful ranking ability, but the raw predicted probabilities may not be perfectly calibrated. Before moving to SHAP explanations and dashboard implementation, this section tests probability calibration and selects a practical operating threshold for discharge-planning use.

This step is important because the threshold is not only a technical setting. It controls the trade-off between identifying more true readmissions and increasing the number of patients flagged for follow-up.

In [ ]:
# ============================================================
# Create calibration validation split from XGBoost training data
# ============================================================

from sklearn.model_selection import train_test_split

X_train_sub, X_calib, y_train_sub, y_calib = train_test_split(
    X_train_xgb,
    y_train,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_train
)

print("X_train_sub:", X_train_sub.shape)
print("X_calib:", X_calib.shape)
print("X_test_xgb:", X_test_xgb.shape)

print("\nCalibration target distribution:")
display(
    y_calib.value_counts()
    .to_frame(name="count")
    .assign(rate=y_calib.value_counts(normalize=True).round(4))
)

In [ ]:
# ============================================================
# Refit XGBoost on train_sub only for calibration testing
# ============================================================

scale_pos_weight_sub = (y_train_sub == 0).sum() / (y_train_sub == 1).sum()

xgb_for_calibration = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight_sub,
    **best_params_map["XGBoost CPU + RFE"]
)

xgb_for_calibration.fit(X_train_sub, y_train_sub)

print("XGBoost refitted for calibration testing.")
print("scale_pos_weight_sub:", round(scale_pos_weight_sub, 4))

In [ ]:
# ============================================================
# Probability Calibration: Original vs Sigmoid vs Isotonic
# ============================================================

from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss, roc_auc_score, average_precision_score

calibrated_sigmoid = CalibratedClassifierCV(
    estimator=xgb_for_calibration,
    method="sigmoid",
    cv="prefit"
)

calibrated_isotonic = CalibratedClassifierCV(
    estimator=xgb_for_calibration,
    method="isotonic",
    cv="prefit"
)

calibrated_sigmoid.fit(X_calib, y_calib)
calibrated_isotonic.fit(X_calib, y_calib)

models_to_compare = {
    "Original XGBoost": best_xgb_model,
    "Calibration-Test XGBoost": xgb_for_calibration,
    "Calibrated Sigmoid": calibrated_sigmoid,
    "Calibrated Isotonic": calibrated_isotonic
}

calibration_results = []

for name, model in models_to_compare.items():
    proba = model.predict_proba(X_test_xgb)[:, 1]

    calibration_results.append({
        "Model": name,
        "ROC-AUC": roc_auc_score(y_test, proba),
        "PR-AUC": average_precision_score(y_test, proba),
        "Brier Score": brier_score_loss(y_test, proba)
    })

calibration_results_df = pd.DataFrame(calibration_results).round(4)
display(calibration_results_df)

calibration_results_df.to_csv(
    OUTPUT_DIR / "xgboost_calibration_model_comparison.csv",
    index=False
)

In [ ]:
# ============================================================
# Calibration Curve Comparison
# ============================================================

plt.figure(figsize=(8, 6))

for name, model in models_to_compare.items():
    proba = model.predict_proba(X_test_xgb)[:, 1]

    prob_true, prob_pred = calibration_curve(
        y_test,
        proba,
        n_bins=10,
        strategy="quantile"
    )

    brier = brier_score_loss(y_test, proba)
    plt.plot(prob_pred, prob_true, marker="o", label=f"{name} | Brier={brier:.3f}")

plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")

plt.title("Calibration Curve Comparison - XGBoost")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed readmission rate")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(OUTPUT_DIR / "xgboost_calibration_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# Select final probability source for dashboard outputs
# ============================================================

# Final decision:
# Sigmoid calibration strongly improves Brier Score
# while keeping ROC-AUC and PR-AUC almost unchanged.

final_model_for_dashboard = calibrated_sigmoid
final_probability_source_name = "Calibrated Sigmoid"

final_proba = final_model_for_dashboard.predict_proba(X_test_xgb)[:, 1]

print("Final probability source selected:", final_probability_source_name)
print("Min probability:", round(final_proba.min(), 4))
print("Max probability:", round(final_proba.max(), 4))
print("Mean probability:", round(final_proba.mean(), 4))

In [ ]:
final_model_for_dashboard = calibrated_sigmoid
final_probability_source_name = "Calibrated Sigmoid"
final_proba = final_model_for_dashboard.predict_proba(X_test_xgb)[:, 1]

In [ ]:
# ============================================================
# Final Threshold Optimization
# ============================================================

thresholds = np.arange(0.10, 0.81, 0.05)

threshold_rows = []

for t in thresholds:
    y_pred_t = (final_proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()

    threshold_rows.append({
        "Threshold": round(t, 2),
        "Accuracy": accuracy_score(y_test, y_pred_t),
        "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred_t),
        "Precision": precision_score(y_test, y_pred_t, zero_division=0),
        "Recall": recall_score(y_test, y_pred_t, zero_division=0),
        "F1-score": f1_score(y_test, y_pred_t, zero_division=0),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "Patients Flagged": tp + fp,
        "Flagged Rate %": (tp + fp) / len(y_test) * 100
    })

final_threshold_df = pd.DataFrame(threshold_rows).round(4)
display(final_threshold_df)

final_threshold_df.to_csv(
    OUTPUT_DIR / "final_threshold_optimization.csv",
    index=False
)

In [ ]:
# ============================================================
# Identify candidate operating thresholds
# ============================================================

best_f1_row = final_threshold_df.loc[
    final_threshold_df["F1-score"].idxmax()
]

best_balanced_acc_row = final_threshold_df.loc[
    final_threshold_df["Balanced Accuracy"].idxmax()
]

practical_candidates = final_threshold_df[
    (final_threshold_df["Flagged Rate %"] >= 20) &
    (final_threshold_df["Flagged Rate %"] <= 40)
].copy()

if len(practical_candidates) > 0:
    best_practical_row = practical_candidates.loc[
        practical_candidates["F1-score"].idxmax()
    ]
else:
    best_practical_row = None

print("Best F1 threshold:")
display(best_f1_row.to_frame().T)

print("Best Balanced Accuracy threshold:")
display(best_balanced_acc_row.to_frame().T)

print("Best Practical threshold with 20%-40% flagged rate:")
if best_practical_row is not None:
    display(best_practical_row.to_frame().T)
else:
    print("No threshold found in the 20%-40% flagged-rate range.")

In [ ]:
# ============================================================
# Create dashboard-friendly risk categories
# Based on calibrated probabilities
# ============================================================

OPERATING_THRESHOLD = 0.15

def assign_risk_category(prob):
    if prob < 0.10:
        return "Low Risk"
    elif prob < OPERATING_THRESHOLD:
        return "Medium Risk"
    else:
        return "High Risk"

dashboard_predictions = X_test_xgb.copy()

dashboard_predictions["actual_readmitted_30d"] = y_test.values
dashboard_predictions["predicted_probability_calibrated"] = final_proba
dashboard_predictions["risk_category"] = dashboard_predictions["predicted_probability_calibrated"].apply(assign_risk_category)

dashboard_predictions["predicted_class"] = (
    dashboard_predictions["predicted_probability_calibrated"] >= OPERATING_THRESHOLD
).astype(int)

risk_summary = (
    dashboard_predictions
    .groupby("risk_category")
    .agg(
        patients=("actual_readmitted_30d", "count"),
        actual_readmissions=("actual_readmitted_30d", "sum"),
        observed_readmission_rate=("actual_readmitted_30d", "mean"),
        avg_predicted_probability=("predicted_probability_calibrated", "mean")
    )
    .reset_index()
)

risk_summary["observed_readmission_rate"] *= 100
risk_summary["avg_predicted_probability"] *= 100

risk_summary = risk_summary.round(2)

display(risk_summary)

dashboard_predictions.to_csv(
    OUTPUT_DIR / "dashboard_patient_risk_predictions.csv",
    index=False
)

risk_summary.to_csv(
    OUTPUT_DIR / "dashboard_risk_category_summary.csv",
    index=False
)

### Probability Calibration and Final Threshold Decision

The original XGBoost model showed useful ranking ability, but its raw predicted probabilities were poorly calibrated. This means that the model could rank patients by risk, but the absolute probability values were not reliable enough for dashboard interpretation.

To address this, probability calibration was tested using sigmoid and isotonic calibration. Sigmoid calibration reduced the Brier Score from 0.2227 to 0.0966 while keeping ROC-AUC and PR-AUC almost unchanged. Therefore, the calibrated sigmoid model was selected as the final probability source for dashboard risk scoring.

After calibration, the probability scale changed. As a result, the default threshold of 0.5 was no longer suitable. Threshold optimization showed that 0.15 provides the best practical balance for discharge planning. At this threshold, the model flags around 24.3% of patients, captures 43.8% of true 30-day readmissions, and achieves the highest F1-score among the tested thresholds.

This threshold is interpreted as an operational decision rather than a purely technical setting. A lower threshold would identify more readmissions but would create too many alerts for discharge planners. A higher threshold would reduce workload but miss more high-risk patients.

For the dashboard, calibrated probabilities will be used to generate patient risk scores and risk categories. SHAP explanations will still be generated from the original XGBoost model, because SHAP explains the tree-based model structure rather than the calibration wrapper.

### Final Dashboard Risk Category Validation

After applying sigmoid probability calibration, the model produced well-calibrated risk groups for dashboard use. The observed readmission rates closely matched the average predicted probabilities within each risk category.

The Low Risk group contained 10,663 patients with an observed 30-day readmission rate of 6.75%, closely matching the average predicted probability of 6.76%. The Medium Risk group contained 4,378 patients with an observed readmission rate of 12.61%, compared with an average predicted probability of 12.32%. The High Risk group contained 4,827 patients with an observed readmission rate of 20.53%, closely matching the average predicted probability of 20.88%.

This confirms that the calibrated model can separate patients into clinically meaningful risk tiers. The High Risk group represents around one quarter of the test population but has approximately three times the readmission rate of the Low Risk group. Therefore, the calibrated probabilities are suitable for dashboard risk scoring and discharge-planning prioritisation.

The final dashboard configuration is:

- Probability source: Calibrated Sigmoid XGBoost probabilities
- Operating threshold: 0.15
- Low Risk: predicted probability below 0.10
- Medium Risk: predicted probability from 0.10 to below 0.15
- High Risk: predicted probability of 0.15 or higher

This threshold should be interpreted as an operational decision. It balances the need to identify high-risk patients with the practical workload of the discharge planning team.

## 12. SHAP Explainability for XGBoost

This section explains how the selected XGBoost model makes readmission risk predictions. The calibrated sigmoid model is used for dashboard probability scoring, but SHAP is generated from the original XGBoost model because SHAP explains the tree-based decision structure.

SHAP values show which features increase or decrease a patient's predicted readmission risk. This supports the Discharge Planner use case by turning the model output into understandable patient-level explanations.

In [ ]:
# ============================================================
# SHAP Explainability for Original XGBoost Model
# ============================================================

import shap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# SHAP should explain the original tree model, not the calibrated wrapper
explainer = shap.Explainer(best_xgb_model, X_train_xgb)

shap_values = explainer(X_test_xgb)

print("SHAP values created.")
print("SHAP values shape:", shap_values.values.shape)
print("X_test_xgb shape:", X_test_xgb.shape)

In [ ]:
# ============================================================
# Global SHAP Feature Importance - Bar Plot
# ============================================================

plt.figure()
shap.plots.bar(
    shap_values,
    max_display=20,
    show=False
)

plt.title("Global SHAP Feature Importance - XGBoost")
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "shap_global_feature_importance_bar.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# SHAP Beeswarm Plot
# ============================================================

plt.figure()
shap.plots.beeswarm(
    shap_values,
    max_display=20,
    show=False
)

plt.title("SHAP Beeswarm Plot - XGBoost")
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "shap_beeswarm_plot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# Plain-English Feature Label Mapping
# ============================================================

feature_label_map = {
    "number_inpatient": "Previous inpatient visits",
    "number_emergency": "Previous emergency visits",
    "number_outpatient": "Previous outpatient visits",
    "prior_utilization_total": "Total prior healthcare use",
    "num_medications": "Medication burden",
    "number_diagnoses": "Number of diagnoses",
    "time_in_hospital": "Length of hospital stay",
    "num_lab_procedures": "Number of lab procedures",
    "num_procedures": "Number of procedures",
    "age_numeric": "Patient age",
    "medication_changed": "Medication changed during admission",
    "diabetes_medication_used": "Diabetes medication used",
    "hba1c_tested": "HbA1c test performed",
    "hba1c_high": "High HbA1c result",
    "max_glu_tested": "Max glucose test performed",
    "max_glu_high": "High max glucose result",
    "is_male": "Male patient",
}

def clean_feature_name(feature):
    """
    Convert encoded feature names into more readable dashboard labels.
    """
    if feature in feature_label_map:
        return feature_label_map[feature]

    cleaned = feature

    cleaned = cleaned.replace("diag_1_group_", "Primary diagnosis: ")
    cleaned = cleaned.replace("diag_2_group_", "Secondary diagnosis: ")
    cleaned = cleaned.replace("diag_3_group_", "Additional diagnosis: ")

    cleaned = cleaned.replace("discharge_group_", "Discharge destination: ")
    cleaned = cleaned.replace("admission_type_group_", "Admission type: ")
    cleaned = cleaned.replace("admission_source_group_", "Admission source: ")
    cleaned = cleaned.replace("medical_specialty_grouped_", "Medical specialty: ")

    cleaned = cleaned.replace("insulin_", "Insulin status: ")
    cleaned = cleaned.replace("metformin_", "Metformin status: ")
    cleaned = cleaned.replace("glipizide_", "Glipizide status: ")
    cleaned = cleaned.replace("glyburide_", "Glyburide status: ")
    cleaned = cleaned.replace("pioglitazone_", "Pioglitazone status: ")
    cleaned = cleaned.replace("rosiglitazone_", "Rosiglitazone status: ")

    cleaned = cleaned.replace("_", " ")

    return cleaned

feature_mapping_df = pd.DataFrame({
    "raw_feature": X_test_xgb.columns,
    "plain_english_label": [clean_feature_name(col) for col in X_test_xgb.columns]
})

display(feature_mapping_df.head(20))

feature_mapping_df.to_csv(
    OUTPUT_DIR / "feature_label_mapping.csv",
    index=False
)

In [ ]:
# ============================================================
# Patient-Level SHAP Explanation Function
# ============================================================

def explain_patient_shap(patient_position, top_n=3):
    """
    Explain one patient prediction using SHAP values.

    patient_position is the row position inside X_test_xgb, not the original dataframe index.
    """
    patient_features = X_test_xgb.iloc[patient_position]
    patient_shap_values = shap_values.values[patient_position]

    explanation_df = pd.DataFrame({
        "feature": X_test_xgb.columns,
        "feature_label": [clean_feature_name(col) for col in X_test_xgb.columns],
        "feature_value": patient_features.values,
        "shap_value": patient_shap_values
    })

    explanation_df["absolute_shap"] = explanation_df["shap_value"].abs()
    explanation_df["effect_direction"] = np.where(
        explanation_df["shap_value"] > 0,
        "Increases readmission risk",
        "Decreases readmission risk"
    )

    explanation_df = explanation_df.sort_values(
        "absolute_shap",
        ascending=False
    )

    return explanation_df.head(top_n)

In [ ]:
# ============================================================
# Example: Explain One High-Risk Patient
# ============================================================

# Use calibrated probabilities for choosing a dashboard high-risk patient
high_risk_positions = np.where(final_proba >= OPERATING_THRESHOLD)[0]

print("Number of high-risk patients:", len(high_risk_positions))

example_patient_position = high_risk_positions[0]

print("Example patient position:", example_patient_position)
print("Calibrated readmission probability:", round(final_proba[example_patient_position], 4))
print("Actual readmission:", y_test.iloc[example_patient_position])

example_explanation = explain_patient_shap(
    patient_position=example_patient_position,
    top_n=5
)

display(example_explanation)

In [ ]:
# ============================================================
# Export Top 3 SHAP Explanations for Dashboard
# ============================================================

patient_explanation_rows = []

for patient_position in range(len(X_test_xgb)):
    patient_id = X_test_xgb.index[patient_position]

    patient_probability = final_proba[patient_position]
    actual_value = y_test.iloc[patient_position]

    if patient_probability < 0.10:
        risk_category = "Low Risk"
    elif patient_probability < OPERATING_THRESHOLD:
        risk_category = "Medium Risk"
    else:
        risk_category = "High Risk"

    top_explanations = explain_patient_shap(
        patient_position=patient_position,
        top_n=3
    )

    row = {
        "patient_position": patient_position,
        "test_index": patient_id,
        "actual_readmitted_30d": actual_value,
        "predicted_probability_calibrated": patient_probability,
        "risk_category": risk_category,
        "predicted_class": int(patient_probability >= OPERATING_THRESHOLD)
    }

    for i, (_, exp_row) in enumerate(top_explanations.iterrows(), start=1):
        row[f"top_{i}_feature"] = exp_row["feature"]
        row[f"top_{i}_label"] = exp_row["feature_label"]
        row[f"top_{i}_value"] = exp_row["feature_value"]
        row[f"top_{i}_shap_value"] = exp_row["shap_value"]
        row[f"top_{i}_effect"] = exp_row["effect_direction"]

    patient_explanation_rows.append(row)

patient_level_shap_df = pd.DataFrame(patient_explanation_rows)

display(patient_level_shap_df.head())

patient_level_shap_df.to_csv(
    OUTPUT_DIR / "patient_level_shap_explanations.csv",
    index=False
)

print("Saved patient-level SHAP explanations.")

In [ ]:
# ============================================================
# Dashboard Example: High-Risk Patient Explanations
# ============================================================

high_risk_examples = (
    patient_level_shap_df
    [patient_level_shap_df["risk_category"] == "High Risk"]
    .sort_values("predicted_probability_calibrated", ascending=False)
    .head(10)
)

display(high_risk_examples)

high_risk_examples.to_csv(
    OUTPUT_DIR / "high_risk_patient_shap_examples.csv",
    index=False
)

### SHAP Explainability Interpretation

SHAP analysis was used to explain the original XGBoost model selected for readmission prediction. While calibrated sigmoid probabilities are used for dashboard risk scoring, SHAP values were generated from the original XGBoost model because SHAP explains the internal tree-based decision structure.

The global SHAP plots identify which features have the strongest overall influence on readmission risk prediction. These features help explain the model at a population level and are useful for technical reporting and managerial interpretation.

Patient-level SHAP explanations were also generated for dashboard use. For each patient, the top three contributing factors were extracted and translated into plain-English labels. Positive SHAP values indicate features that increase predicted readmission risk, while negative SHAP values indicate features that reduce predicted risk.

This supports the Discharge Planner workflow by moving the system beyond a simple risk score. Instead of only showing that a patient is high risk, the dashboard can explain why the patient was flagged, allowing more targeted interventions such as medication review, follow-up scheduling, or care-transition support.